In [3]:
!pip install langchain langchain-anthropic langgraph anthropic pandas scikit-learn openpyxl -q

In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import anthropic

from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langgraph.prebuilt import create_react_agent

os.environ["ANTHROPIC_API_KEY"] = ""

llm = ChatAnthropic(model="claude-opus-4-5", max_tokens=1000)
print("Setup complete.")

Setup complete.


In [6]:
# Upload swiftpay_fintech_dataset.xlsx to Colab first if not already there
FILE_PATH = "/content/swiftpay_fintech_dataset.xlsx"

xl = pd.ExcelFile(FILE_PATH)
df_pl      = pd.read_excel(xl, sheet_name="P&L Full",    header=2)
df_risk    = pd.read_excel(xl, sheet_name="Risk & Fraud", header=2)
df_rev     = pd.read_excel(xl, sheet_name="Revenue Bridge", header=2)

# Set index to first column for easy row lookup
df_pl.set_index(df_pl.columns[0],   inplace=True)
df_risk.set_index(df_risk.columns[0], inplace=True)
df_rev.set_index(df_rev.columns[0],  inplace=True)

print("Sheets loaded:")
print(f"  P&L rows: {len(df_pl)}")
print(f"  Risk rows: {len(df_risk)}")
print(f"  Revenue rows: {len(df_rev)}")

Sheets loaded:
  P&L rows: 33
  Risk rows: 24
  Revenue rows: 19


In [7]:
MONTHS = ["Jan-24","Feb-24","Mar-24","Apr-24","May-24","Jun-24",
          "Jul-24","Aug-24","Sep-24","Oct-24","Nov-24","Dec-24"]

@tool
def get_revenue_trend(metric: str = "Net Revenue") -> str:
    """Get monthly values for a revenue or P&L metric from the financial data."""
    try:
        row = df_pl.loc[df_pl.index.str.contains(metric, case=False, na=False)].iloc[0]
        vals = pd.to_numeric(row[MONTHS], errors="coerce").dropna()
        result = {m: round(v, 1) for m, v in zip(MONTHS, vals)}
        return f"{metric} monthly values: {result}"
    except:
        return f"Could not find metric: {metric}"

@tool
def detect_anomalies(metric: str = "Net Revenue") -> str:
    """Detect statistical anomalies (Z-score > 2) in a financial metric."""
    try:
        row = df_pl.loc[df_pl.index.str.contains(metric, case=False, na=False)].iloc[0]
        vals = pd.to_numeric(row[MONTHS], errors="coerce").dropna()
        mean, std = vals.mean(), vals.std()
        anomalies = []
        for month, val in zip(MONTHS, vals):
            z = abs((val - mean) / std) if std > 0 else 0
            if z > 2.0:
                anomalies.append(f"{month}: {round(val,1)} (Z={round(z,2)})")
        if anomalies:
            return f"Anomalies in {metric}: {'; '.join(anomalies)}"
        return f"No anomalies detected in {metric}"
    except:
        return f"Could not analyse metric: {metric}"

@tool
def forecast_metric(metric: str = "Net Revenue") -> str:
    """Forecast the next 3 months for a financial metric using linear regression."""
    try:
        row = df_pl.loc[df_pl.index.str.contains(metric, case=False, na=False)].iloc[0]
        vals = pd.to_numeric(row[MONTHS], errors="coerce").dropna().values
        X = np.arange(1, len(vals)+1).reshape(-1,1)
        model = LinearRegression().fit(X, vals)
        future = model.predict(np.arange(len(vals)+1, len(vals)+4).reshape(-1,1))
        return (f"{metric} forecast: "
                f"Jan-25={round(future[0],1)}, "
                f"Feb-25={round(future[1],1)}, "
                f"Mar-25={round(future[2],1)}")
    except:
        return f"Could not forecast metric: {metric}"

@tool
def get_risk_metrics() -> str:
    """Get key risk and fraud metrics from the Risk & Fraud sheet."""
    try:
        risk_keys = ["Overall Fraud Rate","Chargeback Rate",
                     "Authorization Rate","30+ DPD","NPL"]
        results = []
        for key in risk_keys:
            matches = df_risk.index.str.contains(key, case=False, na=False)
            if matches.any():
                row = df_risk.loc[matches].iloc[0]
                vals = pd.to_numeric(row[MONTHS], errors="coerce").dropna()
                avg = round(vals.mean(), 5)
                latest = round(vals.iloc[-1], 5)
                results.append(f"{key}: avg={avg}, latest={latest}")
        return "Risk metrics:\n" + "\n".join(results)
    except Exception as e:
        return f"Error retrieving risk metrics: {str(e)}"

tools = [get_revenue_trend, detect_anomalies, forecast_metric, get_risk_metrics]
print(f"✅ {len(tools)} tools registered.")

✅ 4 tools registered.


In [8]:
from langchain_core.messages import HumanMessage

# System prompt defines the agent's role and behaviour
system_prompt = """You are a senior fintech financial analyst with access to SwiftPay Corp's
financial data. You have tools to:
- Retrieve revenue and P&L trends
- Detect statistical anomalies
- Forecast future performance
- Retrieve risk and fraud metrics

When asked a question, use the appropriate tools to gather data first,
then synthesise a clear, data-driven answer. Always cite specific numbers."""

agent = create_react_agent(llm, tools, prompt=system_prompt)
print("✅ Agent created.")

✅ Agent created.


/tmp/ipykernel_661/3647215424.py:14: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=system_prompt)


In [9]:
def ask_agent(question):
    print(f"❓ Question: {question}")
    print("-" * 50)
    result = agent.invoke({"messages": [HumanMessage(content=question)]})
    answer = result["messages"][-1].content
    print(f"🤖 Answer:\n{answer}")
    print("=" * 50)
    return answer

# Test with 3 progressively complex questions
ask_agent("What is the revenue trend for SwiftPay and are there any anomalies?")

❓ Question: What is the revenue trend for SwiftPay and are there any anomalies?
--------------------------------------------------
🤖 Answer:
## SwiftPay Revenue Trend Analysis

### Monthly Revenue Performance (2024)

| Month | Net Revenue ($M) | MoM Change |
|-------|------------------|------------|
| Jan-24 | $300.0M | — |
| Feb-24 | $316.9M | +5.6% |
| Mar-24 | $304.0M | -4.1% |
| Apr-24 | $346.4M | +13.9% |
| May-24 | $379.8M | +9.6% |
| Jun-24 | $419.5M | +10.5% |
| Jul-24 | $452.8M | +7.9% |
| Aug-24 | $363.7M | **-19.7%** |
| Sep-24 | $476.7M | +31.1% |
| Oct-24 | $512.1M | +7.4% |
| Nov-24 | $551.1M | +7.6% |
| Dec-24 | $605.2M | +9.8% |

### Key Insights

1. **Strong Overall Growth**: SwiftPay's net revenue grew from **$300.0M in January** to **$605.2M in December 2024**, representing a **101.7% increase** over the year.

2. **Average Monthly Growth**: Approximately **+6.6% month-over-month** on average.

3. **Notable Fluctuation in August**: There was a significant dip in Augu

"## SwiftPay Revenue Trend Analysis\n\n### Monthly Revenue Performance (2024)\n\n| Month | Net Revenue ($M) | MoM Change |\n|-------|------------------|------------|\n| Jan-24 | $300.0M | — |\n| Feb-24 | $316.9M | +5.6% |\n| Mar-24 | $304.0M | -4.1% |\n| Apr-24 | $346.4M | +13.9% |\n| May-24 | $379.8M | +9.6% |\n| Jun-24 | $419.5M | +10.5% |\n| Jul-24 | $452.8M | +7.9% |\n| Aug-24 | $363.7M | **-19.7%** |\n| Sep-24 | $476.7M | +31.1% |\n| Oct-24 | $512.1M | +7.4% |\n| Nov-24 | $551.1M | +7.6% |\n| Dec-24 | $605.2M | +9.8% |\n\n### Key Insights\n\n1. **Strong Overall Growth**: SwiftPay's net revenue grew from **$300.0M in January** to **$605.2M in December 2024**, representing a **101.7% increase** over the year.\n\n2. **Average Monthly Growth**: Approximately **+6.6% month-over-month** on average.\n\n3. **Notable Fluctuation in August**: There was a significant dip in August 2024 ($363.7M, down 19.7% from July), followed by a strong rebound in September (+31.1%). While this is a notabl

In [10]:
ask_agent(
    "What are the top risks for SwiftPay right now? "
    "Check both the fraud metrics and any revenue anomalies, "
    "then give me a forecast for the next 3 months."
)

❓ Question: What are the top risks for SwiftPay right now? Check both the fraud metrics and any revenue anomalies, then give me a forecast for the next 3 months.
--------------------------------------------------
🤖 Answer:
## SwiftPay Risk Assessment & Outlook

### 🛡️ Fraud & Risk Metrics Summary

| Metric | Average | Latest | Status |
|--------|---------|--------|--------|
| **Fraud Rate** | 0.152% | 0.09% | ✅ Improving |
| **Chargeback Rate** | 0.065% | 0.03% | ✅ Improving |
| **Authorization Rate** | 98.5% | 98.95% | ✅ Strong |
| **Non-Performing Loans (NPL)** | 1.13% | 0.70% | ✅ Improving |

**Key Takeaways on Risk:**
- **Fraud rate has dropped significantly** — from an average of 0.152% down to 0.09% in the latest period (a ~41% improvement). This suggests fraud controls are working effectively.
- **Chargebacks are well below average** — at 0.03% vs. 0.065% average, indicating good transaction quality and dispute management.
- **NPL trending down** — at 0.70% vs. 1.13% average, wh

"## SwiftPay Risk Assessment & Outlook\n\n### 🛡️ Fraud & Risk Metrics Summary\n\n| Metric | Average | Latest | Status |\n|--------|---------|--------|--------|\n| **Fraud Rate** | 0.152% | 0.09% | ✅ Improving |\n| **Chargeback Rate** | 0.065% | 0.03% | ✅ Improving |\n| **Authorization Rate** | 98.5% | 98.95% | ✅ Strong |\n| **Non-Performing Loans (NPL)** | 1.13% | 0.70% | ✅ Improving |\n\n**Key Takeaways on Risk:**\n- **Fraud rate has dropped significantly** — from an average of 0.152% down to 0.09% in the latest period (a ~41% improvement). This suggests fraud controls are working effectively.\n- **Chargebacks are well below average** — at 0.03% vs. 0.065% average, indicating good transaction quality and dispute management.\n- **NPL trending down** — at 0.70% vs. 1.13% average, which is positive for credit portfolio health.\n- **Authorization rates are high** at 98.95%, meaning smooth payment processing with minimal declines.\n\n### 📊 Revenue Anomaly Check\n\n**No statistical anomalie